# Autonomous Research Agent

Framework: LangGraph | LLM: Groq LLaMA 3.1 | Search: Serper API

Workflow: User Query -> Planner -> Web Search -> Retrieval -> Summarization -> Report

| Agent | Role |
|---|---|
| Planner | Breaks topic into sub-questions |
| Web Search | Searches using Serper API |
| Retrieval | Embeds and retrieves with FAISS |
| Summarization | Condenses findings |
| Report Writer | Generates structured report |

## Step 1 - Install and Import Libraries

In [3]:
!pip install requests==2.32.4 --force-reinstall

  Using cached requests-2.32.4-py3-none-any.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.4.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (39 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.32.4-py3-none-any.whl (64 kB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
Using cached charset_normalizer-3.4.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (196 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.6.3
    Uninstalling urllib3-2.6.3:
      Successfully uninstalled urllib3-2.6.3
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Su

In [1]:
!pip install -q langgraph langchain langchain-groq langchain-community langchain-huggingface
!pip install -q faiss-cpu sentence-transformers gradio requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import os, json, time, requests, warnings
warnings.filterwarnings('ignore')
from typing import TypedDict, List, Annotated
import operator
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import gradio as gr
print('All libraries imported!')

All libraries imported!


## Step 2 - API Keys (Groq + Serper)

In [6]:
import getpass
import os
from google.colab import userdata # Keep this import for the try block

try:
    # Assuming the API keys are stored under these names in Colab Secrets
    GROQ_API_KEY   = userdata.get('GROQ_API_KEY')
    SERPER_API_KEY = userdata.get('SERPER_API_KEY')
    if GROQ_API_KEY and SERPER_API_KEY:
        print('Both keys loaded from Colab Secrets')
    else:
        # Fallback to getpass if not found in secrets (e.g., if userdata.get returns None)
        raise ValueError('API keys not found in Colab Secrets with expected names.')
except (ImportError, ValueError, userdata.SecretNotFoundError): # Catch SecretNotFoundError as well
    print('Colab Secrets not available or keys not found. Please enter them manually.')
    GROQ_API_KEY   = getpass.getpass('Groq API Key: ')
    SERPER_API_KEY = getpass.getpass('Serper API Key (free at serper.dev): ')

os.environ['GROQ_API_KEY']   = GROQ_API_KEY
os.environ['SERPER_API_KEY'] = SERPER_API_KEY
print('API keys configured!')
print(f'Groq API Key length: {len(GROQ_API_KEY) if GROQ_API_KEY else 0}')
print(f'Serper API Key length: {len(SERPER_API_KEY) if SERPER_API_KEY else 0}')

Colab Secrets not available or keys not found. Please enter them manually.
Groq API Key: ··········
Serper API Key (free at serper.dev): ··········
API keys configured!
Groq API Key length: 56
Serper API Key length: 40


## Step 3 - Select Groq Model

In [8]:
SELECTED = 'llama-3.1-8b-instant'
llm = ChatGroq(api_key=GROQ_API_KEY, model_name=SELECTED, temperature=0.3, max_tokens=2048)
test = llm.invoke('Reply in 5 words: system ready.')
print(f'Model : {SELECTED}')
print(f'Test  : {test.content}')

Model : llama-3.1-8b-instant
Test  : Affirmative, all systems online.


## Step 4 - LangGraph Agentic Framework

In [9]:
class ResearchState(TypedDict):
    topic:             str
    research_plan:     List[str]
    search_results:    Annotated[List[dict], operator.add]
    retrieved_content: Annotated[List[str], operator.add]
    summaries:         Annotated[List[str], operator.add]
    final_report:      str
    status_log:        Annotated[List[str], operator.add]

def serper_search(query, n=5):
    try:
        resp = requests.post(
            'https://google.serper.dev/search',
            headers={'X-API-KEY': SERPER_API_KEY, 'Content-Type': 'application/json'},
            json={'q': query, 'num': n}, timeout=15
        )
        data = resp.json()
        return [{'title': r.get('title',''), 'snippet': r.get('snippet',''), 'link': r.get('link','')}
                for r in data.get('organic', [])[:n]]
    except Exception as e:
        return [{'title': 'Result', 'snippet': f'Info about {query}', 'link': '#'}]

print('State and Serper tool defined.')

State and Serper tool defined.


In [10]:
def planner_agent(state):
    print('[Planner] Creating research plan...')
    prompt = (
        f'Break this topic into 4 research sub-questions. '
        f'Return ONLY a JSON array of strings.\n'
        f'Topic: {state["topic"]}\n'
        f'Format: ["q1", "q2", "q3", "q4"]'
    )
    resp = llm.invoke(prompt)
    try:
        txt = resp.content.strip()
        plan = json.loads(txt[txt.find('['):txt.rfind(']')+1])
    except:
        plan = [
            f'What is {state["topic"]}?',
            f'Latest trends in {state["topic"]}',
            f'Key challenges in {state["topic"]}',
            f'Future outlook for {state["topic"]}',
        ]
    for i, q in enumerate(plan, 1):
        print(f'  {i}. {q}')
    return {**state, 'research_plan': plan, 'status_log': [f'Planner: {len(plan)} sub-questions']}

def web_search_agent(state):
    print('[Web Search] Searching...')
    results = []
    for q in state['research_plan']:
        results.extend(serper_search(q, n=3))
        time.sleep(0.5)
    print(f'  Total results: {len(results)}')
    return {**state, 'search_results': results, 'status_log': [f'Web Search: {len(results)} results']}

def retrieval_agent(state):
    print('[Retrieval] Building FAISS index...')
    docs = [Document(page_content=f"{r['title']}\n{r['snippet']}", metadata={'source': r['link']})
            for r in state['search_results']]
    emb  = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2',
                                  model_kwargs={'device': 'cpu'})
    vs   = FAISS.from_documents(docs, emb)
    top  = vs.similarity_search(state['topic'], k=min(8, len(docs)))
    content = [d.page_content for d in top]
    print(f'  Indexed {len(docs)}, retrieved top {len(top)}')
    return {**state, 'retrieved_content': content, 'status_log': [f'Retrieval: top {len(top)} docs']}

def summarization_agent(state):
    print('[Summarization] Summarizing...')
    summaries = []
    context   = '\n\n'.join(state['retrieved_content'][:4])
    for i, sq in enumerate(state['research_plan']):
        prompt = f'Summarize in 2 paragraphs answering: "{sq}"\nContext:\n{context[:2500]}\nSummary:'
        resp   = llm.invoke(prompt)
        summaries.append(f'Sub-topic {i+1}: {sq}\n{resp.content}')
        print(f'  Done {i+1}/{len(state["research_plan"])}')
        time.sleep(0.3)
    return {**state, 'summaries': summaries, 'status_log': [f'Summarized {len(summaries)} topics']}

def report_writer_agent(state):
    print('[Report Writer] Writing final report...')
    summaries_text = '\n\n'.join(state['summaries'])
    sources = list({r['link'] for r in state['search_results'] if r['link'] != '#'})
    prompt  = (
        f'Write a professional research report on: "{state["topic"]}"\n\n'
        f'Findings:\n{summaries_text[:4000]}\n\n'
        'Structure: 1) Executive Summary 2) Key Findings 3) Detailed Analysis 4) Implications 5) Conclusions'
    )
    resp   = llm.invoke(prompt)
    report = resp.content
    if sources:
        report += '\n\n---\n## Sources\n' + '\n'.join(f'- {s}' for s in sources[:8])
    print(f'  Report: {len(report.split())} words')
    return {**state, 'final_report': report, 'status_log': [f'Report: {len(report.split())} words']}

print('All 5 agent functions defined.')

All 5 agent functions defined.


In [11]:
wf = StateGraph(ResearchState)
wf.add_node('planner',    planner_agent)
wf.add_node('web_search', web_search_agent)
wf.add_node('retrieval',  retrieval_agent)
wf.add_node('summarize',  summarization_agent)
wf.add_node('report',     report_writer_agent)
wf.set_entry_point('planner')
wf.add_edge('planner',    'web_search')
wf.add_edge('web_search', 'retrieval')
wf.add_edge('retrieval',  'summarize')
wf.add_edge('summarize',  'report')
wf.add_edge('report',     END)
app = wf.compile()
print('LangGraph compiled! Flow: Planner -> Web Search -> Retrieval -> Summarize -> Report')

LangGraph compiled! Flow: Planner -> Web Search -> Retrieval -> Summarize -> Report


In [12]:
# Quick test run
init = {
    'topic': 'Generative AI impact on healthcare 2024',
    'research_plan': [], 'search_results': [], 'retrieved_content': [],
    'summaries': [], 'final_report': '', 'status_log': []
}
t0 = time.time()
result = app.invoke(init)
print(f'Total time: {time.time()-t0:.1f}s')
print('\nExecution log:')
for log in result['status_log']:
    print(f'  {log}')
print('\nReport preview (first 800 chars):')
print(result['final_report'][:800])

[Planner] Creating research plan...
  1. What are the current applications of Generative AI in healthcare?
  2. How does Generative AI improve patient outcomes and care quality in 2024?
  3. What are the potential risks and challenges associated with the adoption of Generative AI in healthcare?
  4. How can healthcare organizations effectively integrate Generative AI into their existing systems and workflows?
[Web Search] Searching...
  Total results: 12
[Retrieval] Building FAISS index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Indexed 12, retrieved top 8
[Summarization] Summarizing...
  Done 1/4
  Done 2/4
  Done 3/4
  Done 4/4
[Report Writer] Writing final report...
  Report: 974 words
Total time: 30.3s

Execution log:
  Planner: 4 sub-questions
  Web Search: 12 results
  Retrieval: top 8 docs
  Summarized 4 topics
  Report: 974 words

Report preview (first 800 chars):
**Generative AI Impact on Healthcare 2024: A Research Report**

**Executive Summary**

This research report explores the current and potential applications of Generative AI in healthcare, its impact on patient outcomes and care quality, and the associated risks and challenges. Our findings indicate that Generative AI has the potential to transform the healthcare industry by enhancing administrative efficiency, clinical productivity, and patient engagement. However, the adoption of Generative AI in healthcare also poses several risks and challenges, including bias in AI-generated data, compromised patient confidentiality, and the need for he

## Step 5 - Gradio Professional Product UI

In [13]:
TOPICS = {
    'Market Research':          'Global AI chip market competitive landscape 2024',
    'Competitive Intelligence': 'Cloud providers AWS Azure Google Cloud comparison',
    'Policy Research':          'EU AI Act impact on technology companies 2024',
    'Healthcare AI':            'Generative AI applications in drug discovery 2024',
    'Finance':                  'Blockchain adoption in banking sector 2024',
    'Custom Topic':             '',
}

def run_agent(topic, use_case, progress=gr.Progress()):
    if use_case != 'Custom Topic':
        topic = TOPICS.get(use_case, topic)
    if not topic.strip():
        return 'Please enter a research topic.', '', '', ''
    progress(0.1, desc='Planner working...')
    state = {
        'topic': topic, 'research_plan': [], 'search_results': [],
        'retrieved_content': [], 'summaries': [], 'final_report': '', 'status_log': []
    }
    try:
        progress(0.4, desc='Searching and retrieving...')
        out = app.invoke(state)
        progress(0.9, desc='Writing report...')
        plan_md = '### Research Plan\n' + '\n'.join(f'**{i+1}.** {q}' for i,q in enumerate(out['research_plan']))
        log_md  = '### Agent Log\n'     + '\n'.join(f'- {l}' for l in out['status_log'])
        srcs    = list({r['link'] for r in out['search_results'] if r['link'] != '#'})[:6]
        src_md  = '### Sources\n'       + '\n'.join(f'- {s}' for s in srcs)
        progress(1.0, desc='Done!')
        return out['final_report'], plan_md, log_md, src_md
    except Exception as e:
        return f'Error: {e}', '', '', ''

CSS = '.gradio-container { background: #f0f4f8 !important; }'

with gr.Blocks(css=CSS, title='Autonomous Research Agent') as demo:
    gr.Markdown(
        '# Autonomous Research Agent\n'
        'LangGraph | Groq LLaMA 3.1 | Serper Web Search\n\n'
        '**5 Agents:** Planner | Web Search | Retrieval | Summarization | Report Writer'
    )
    with gr.Row():
        with gr.Column(scale=1):
            use_case = gr.Dropdown(choices=list(TOPICS.keys()), value='Custom Topic', label='Industry Use Case')
            t_in     = gr.Textbox(label='Research Topic', placeholder='e.g., Impact of AI on supply chains...', lines=3)
            with gr.Row():
                run_btn = gr.Button('Run Research Agent', variant='primary')
                clr_btn = gr.Button('Clear')
            gr.Markdown('### Quick Templates')
            with gr.Row():
                gr.Button('Market Research').click(
                    lambda: ('Market Research', TOPICS['Market Research']),
                    outputs=[use_case, t_in])
                gr.Button('Policy Research').click(
                    lambda: ('Policy Research', TOPICS['Policy Research']),
                    outputs=[use_case, t_in])
            with gr.Row():
                gr.Button('Healthcare AI').click(
                    lambda: ('Healthcare AI', TOPICS['Healthcare AI']),
                    outputs=[use_case, t_in])
                gr.Button('Competitive Intel').click(
                    lambda: ('Competitive Intelligence', TOPICS['Competitive Intelligence']),
                    outputs=[use_case, t_in])
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab('Final Report'):   rep_out = gr.Markdown()
                with gr.Tab('Research Plan'):  plan_out = gr.Markdown()
                with gr.Tab('Agent Log'):      log_out  = gr.Markdown()
                with gr.Tab('Sources'):        src_out  = gr.Markdown()
    run_btn.click(run_agent, inputs=[t_in, use_case], outputs=[rep_out, plan_out, log_out, src_out])
    clr_btn.click(lambda: ('','','',''), outputs=[rep_out, plan_out, log_out, src_out])
demo.launch(share=True)
print('Research Agent UI launched!')

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2559af58c5e3fbe674.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Research Agent UI launched!
